In [ ]:
import sys
from tqdm import tqdm
import pickle as pkl
import os 
from os import path

import numpy as np
import seaborn as sns
from scipy.stats import rankdata

from ephyslib import preprocessing
from ephyslib import crossvalidation

from macaquethings.data_util.load_data import load_data
from macaquethings.data_util.process_data import process_data
from macaquethings.rdm_util import get_rdm_design_sort_indices

from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedShuffleSplit,
    cross_validate,
)
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels

import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch import optim
from torch.nn.functional import cross_entropy
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import wandb

from datetime import datetime

figure_style()  # set consistent plotting defaults for all figs

now = datetime.now()
date_time = now.strftime("%m-%d-%Y_%H:%M:%S")
savedir = 'rnn_trajectory_decoding/' + "monkeyF_mua_chan" + date_time

os.makedirs(savedir, exist_ok=True)


# Model Definition

In [ ]:
class LabelEncoder:
    def __init__(self, labels):
        self.classes = sorted(set(labels))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def encode(self, labels):
        return torch.tensor(
            [self.class_to_idx[l] for l in labels],
            dtype=torch.long
        )


class TrialDataset(Dataset):
    def __init__(self, X, y, shuffle_time=False):
        """
        X: numpy array [trials, time, dims]
        y: list of strings
        shuffle_time: shuffle time (axis 0) every time a trial is accessed
        """
        assert len(X) == len(y)

        self.X = torch.from_numpy(X).float()
        
        self.label_encoder = LabelEncoder(y)
        self.y = self.label_encoder.encode(y)
        self.shuffle_time = shuffle_time

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.shuffle_time:
            perm = torch.randperm(x.shape[0])
            x = x[perm]
        return x, self.y[idx]
        

class Model(nn.Module):
    def __init__(
        self,
        rnn_params,
        classifier_params,
        proj_dim=32,
        dropout_p=0.4,
        loss_at_last_timestep=True,
        device='cpu',
        patience=1000
    ):
        super().__init__()

        self.loss_at_last_timestep = loss_at_last_timestep
        self.device = device
        self.patience = patience

        input_dim = rnn_params["input_size"]

        # ----- input bottleneck (MOST IMPORTANT PART)
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout_p)
        )

        # ----- GRU instead of LSTM
        rnn_params = rnn_params.copy()
        rnn_params["input_size"] = proj_dim
        self.gru = nn.GRU(**rnn_params)

        # ----- optional normalization after GRU
        self.post_rnn_norm = nn.Sequential(
            nn.Dropout(dropout_p),
            nn.LayerNorm(rnn_params["hidden_size"])
        )

        # ----- classifier
        self.classifier = nn.Linear(**classifier_params)

    def forward(self, x):
        # x: [batch, time, channels]
        b, t, _ = x.shape

        # project inputs
        x = self.input_proj(x)

        # GRU
        rnn_out, _ = self.gru(x)  # [batch, time, hidden]

        # normalize hidden state
        rnn_out = self.post_rnn_norm(rnn_out)

        # flatten time for classification
        rnn_out_flat = rnn_out.reshape(b * t, rnn_out.shape[-1])
        preds_flat = self.classifier(rnn_out_flat)

        preds = preds_flat.reshape(b, t, -1)
        return preds


    def loss_func(self, x_seq, y):
        """
        x_seq has predictions for each time step shape [batch x time x n_classes]
        y has the label indices
        """
        # transpose time last
        preds = torch.moveaxis(x_seq, 1, 2)
    
        # same class for each time step, stack
        ytime = y[:, None].repeat(1, preds.shape[2])
        crossent_per_time = cross_entropy(preds, ytime, reduction='none').mean(axis=0)  # reduce batch dim
        return crossent_per_time


    def compute_losses(self, x, y, return_pred=False):
        x = x.to(self.device)
        y = y.to(self.device)
        preds = self.forward(x)
        losses = self.loss_func(preds, y)
        if not return_pred:
            return losses
        else:
            return losses, preds

    
    def step(self, x, y, step_idx, optimizer):
        device = self.device
        # ---- forward
        optimizer.zero_grad()
        losses = self.compute_losses(x, y)
        
        # ---- loss processing and logging
        self.losses[step_idx] = losses.detach().cpu()

        if self.loss_at_last_timestep:
            loss = losses[-1]
        else:
            loss = losses.mean()

        self.loss[step_idx] = loss.detach().cpu()

        # ---- backward
        loss.backward()
        optimizer.step()

    def validate(self, val_loader):
        losses_per_batch = []
        accs_per_batch = []
        with torch.no_grad():
            self.eval()  # switch off dropout
            
            for xbatch, ybatch in val_loader:
                # --- compute loss
                losses, preds = self.compute_losses(xbatch, ybatch, return_pred=True)
                losses_per_batch.append(losses)
                # --- compute accuracy
                pred_label = torch.argmax(preds.detach().cpu(), dim=2)  # highest logit over class axis
                correct = torch.eq(ybatch[:, None], pred_label)
                acc = correct.sum(axis=0) / correct.shape[0]
                accs_per_batch.append(acc)
        losses_per_batch = torch.stack(losses_per_batch)
        accs_per_batch = torch.stack(accs_per_batch)
        return losses_per_batch.mean(axis=0), accs_per_batch.mean(axis=0)
        

    def train_model(self, train_loader, val_loader, n_epochs, optimizer):
        # set up logging
        nsample, ntime, nchan = train_loader.dataset.X.shape
        nbatch = len(train_loader)
        
        self.losses = torch.zeros((nbatch * n_epochs, ntime))
        self.loss = torch.zeros((nbatch * n_epochs,))
        self.val_losses = torch.zeros((n_epochs, ntime))
        self.val_accs = torch.zeros((n_epochs, ntime))
        self.epochs_since_best = 0

        # store best val acc and weights
        self.best_val = 0
        self.best_val_params = None

        for i_ep in range(n_epochs):
            self.train()  # switch on dropout
            # train
            for i_batch, (xbatch, ybatch) in enumerate(train_loader):
                batch_idx = (nbatch * i_ep) + i_batch
                self.step(xbatch, ybatch, batch_idx, optimizer)

            # validate
            val_losses, val_accs = self.validate(val_loader)
            self.val_losses[i_ep] = val_losses
            self.val_accs[i_ep] = val_accs
            
            train_loss_ep = self.loss[(nbatch * i_ep):(nbatch * (i_ep + 1))].mean()
            val_loss_ep = val_losses[-1] if self.loss_at_last_timestep else val_losses.mean()
            val_acc_ep = val_accs[-1] if self.loss_at_last_timestep else val_accs.mean()
            print(f">>> EPOCH {i_ep + 1} / {n_epochs} - train loss: {train_loss_ep:.4f} - val loss: {val_loss_ep:.4f} - val acc. {val_acc_ep:.4f}", end="\r")

            if val_acc_ep > self.best_val:
                self.best_val = val_acc_ep
                self.best_val_params = self.state_dict().copy()
                self.epochs_since_best = 0
            else:
                self.epochs_since_best += 1

            wandb.log({
                "epoch": i_ep + 1,
                "train_loss": train_loss_ep.item(),
                "val_loss": val_loss_ep.item(),
                "val_accuracy": val_acc_ep.item(),
                "best_val": self.best_val.item()
            })
            print(f">>> EPOCH {i_ep + 1}/{config.epochs} - train loss: {train_loss_ep:.4f} - val loss: {val_loss_ep:.4f} - val acc: {val_acc_ep:.4f} - epochs since best {str(self.epochs_since_best).zfill(4)} / {str(self.patience).zfill(4)} ({self.best_val:.4f})", end="\r")
            
            if self.epochs_since_best > self.patience:
                print('Patience limit reached. Stopping training.')
                break    
                

# zscore per session
def zscore_over_time(X, z_stats=None):
    X_flat = X.reshape(-1, X.shape[2])
    if z_stats is None:
        print('computing z-score')
        mean = X_flat.mean(axis=0, keepdims=True)
        std = X_flat.std(axis=0, keepdims=True)
    else:
        mean, std = z_stats
    # apply
    X_z = (X_flat - mean) / (std + 1e-6)  # avoid divide by zero
    
    # reshape back: [trials, time-1, channels]
    X_z = X_z.reshape(X.shape)
    return X_z, (mean, std)


def save_as_pickle(fpath, data):
    print(fpath)
    with open(fpath, 'wb') as f:
        pkl.dump(data, f)


# Load Dataset

In [ ]:
# --- data config
data_cfg = dict(
    monkey="monkeyF",
    labels="category",
    baseline=0,
    session_ids=np.array([0, 1, 2, 3, 4, 5]),
    array_indices=np.arange(16) + 1,
    # array_indices=np.array([11]),
    rois=np.array([3]),
    good_channel_threshold=1.5,
    session_ids_for_channel_selection=np.array([0, 1, 2, 3, 4, 5]),
    neural_data="mua",
    dataset='allMUA'
)

decode_cfg = dict(
    standardize_data=0,
    trial_averaged=False,  # must be false, for compatibility with data preprocessing
    avg_time=10,  # window size for averaging in time
)

time, X, y, groups, im_number, sess_idx, info, data_strs, h5_handle = load_data(
    data_cfg, root='..'
)
roi_str, arr_str, sess_str = data_strs
X, groups = process_data(X, sess_idx, im_number, groups, decode_cfg)

n_channels = X.shape[1]

# Choose Data

In [ ]:
decoding_ts = np.arange(50, 250, 10)
tmask = np.isin(time, decoding_ts)
Xsel = X[..., tmask]
Xsel = np.moveaxis(Xsel, 1,2)
print(Xsel.shape)

# Model Hyperparameters

In [ ]:
DEVICE = 'cuda'

nlayer = 1
proj_dim = 32
dropout_p = .3
hidden_size = 64
batchsize = 512
patience = 500

rnn_params = dict(
    input_size=n_channels,
    hidden_size=hidden_size,
    num_layers=nlayer,
    bias=True,
    batch_first=True,
    dropout=0.,
)

classifier_params = dict(
    in_features=hidden_size,
    out_features=len(np.unique(y)),
    bias=True
)

EPOCHS = 20000


# Crossvalidated Training

In [ ]:
# compute splits and train
cv = crossvalidation.ImageStratifiedShuffleSplit(5, random_state=42)

perf_per_split = []
for cv_split_idx, (train_idx, test_idx) in enumerate(cv.split(Xsel, y, groups=groups)):

    print("\n" * 10)
    print(f"--------------- STARTING SPLIT {cv_split_idx + 1}")

    # ----- temporal difference
    X_diff = Xsel[:, 1:, :] - Xsel[:, :-1, :]  # [trials, time-1, channels]
    
    # --- z-score for train
    X_diff_train = X_diff[train_idx]
    sess_idx_train = sess_idx[train_idx]
    X_diff_test = X_diff[test_idx]
    sess_idx_test = sess_idx[test_idx]
    
    # -----------------------------------------Z-score per session
    
    X_train = np.zeros_like(X_diff_train)
    X_test = np.zeros_like(X_diff_test)
    X_train.fill(np.nan)
    X_test.fill(np.nan)
    
    unique_sessions = np.unique(sess_idx)
    for s in unique_sessions:
        print(f"session {s}")
        sess_mask_train = sess_idx_train == s
        sess_mask_test = sess_idx_test == s
        
        X_train_sess = X_diff_train[sess_mask_train]
        X_test_sess = X_diff_test[sess_mask_test]
    
        Xtrain_z, z_stats = zscore_over_time(X_train_sess)
        Xtest_z, _ = zscore_over_time(X_test_sess, z_stats)
    
        X_train[sess_mask_train] = Xtrain_z
        X_test[sess_mask_test] = Xtest_z
    
    assert not np.isnan(X_train).any()
    assert not np.isnan(X_test).any()
    
    print(X_train.shape, X_test.shape)
    
    dset_train = TrialDataset(X_train, y[train_idx])
    dset_test = TrialDataset(X_test, y[test_idx])

    # ---------------------------  Initialize a new WandB run
    
    wandb.init(
        project="time-series-classification",
        reinit=True,  # ensures a new run every time you run this cell
        config={
            "proj_dim": proj_dim,
            "gru_hidden": hidden_size,
            "gru_layers": nlayer,
            "dropout_input": dropout_p,
            "batch_size": batchsize,
            "learning_rate": 0.001,
            "weight_decay": 0.01,
            "epochs": EPOCHS,
            "loss_at_last_timestep": True,
            "rnn_params": rnn_params,
            "classifier_params": classifier_params,
            "patience": patience,
            "time bins": "delta"
    }
    )
    
    config = wandb.config
    
    # ------------------------------- setup model
    
    model = Model(
        rnn_params, 
        classifier_params, 
         proj_dim=proj_dim,
        dropout_p=dropout_p,
       loss_at_last_timestep=True, 
        device=DEVICE,
        patience=patience
    )
    
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), weight_decay=0.01, lr=0.001)
    
    dloader_train = DataLoader(  
        dset_train,
        batch_size=batchsize,
        shuffle=True,
    )
    
    dloader_test = DataLoader(  
        dset_test,
        batch_size=batchsize,
        shuffle=False,
    )

    # ------------------------------- train model
    
    model.train_model(dloader_train, dloader_test, EPOCHS, optimizer)
    wandb.finish()   
    
    acc_for_fold = model.best_val
    perf_per_split.append(acc_for_fold)


In [ ]:
perf_per_split = np.array(perf_per_split)
perf_per_split


save_as_pickle(path.join(savedir, 'perf.pkl'), perf_per_split)
save_as_pickle(path.join(savedir, 'config.pkl'), config)


# Rerun without temporal differencing

In [ ]:
decode_cfg['standardize_data'] = 1

time, X, y, groups, im_number, sess_idx, info, data_strs, h5_handle = load_data(
    data_cfg, root='..'
)
roi_str, arr_str, sess_str = data_strs
X, groups = process_data(X, sess_idx, im_number, groups, decode_cfg)

n_channels = X.shape[1]

tmask = np.isin(time, decoding_ts)
Xsel = X[..., tmask]
Xsel = np.moveaxis(Xsel, 1,2)
print(Xsel.shape)

In [ ]:

# compute splits and train
cv = crossvalidation.ImageStratifiedShuffleSplit(5, random_state=42)

perf_per_split_abs_t = []
for cv_split_idx, (train_idx, test_idx) in enumerate(cv.split(Xsel, y, groups=groups)):

    print("\n" * 10)
    print(f"--------------- STARTING SPLIT {cv_split_idx + 1}")

    X_train = Xsel[train_idx]  
    X_test = Xsel[test_idx]
    
    dset_train = TrialDataset(X_train, y[train_idx])
    dset_test = TrialDataset(X_test, y[test_idx])

    # ---------------------------  Initialize a new WandB run
    
    wandb.init(
        project="time-series-classification",
        reinit=True,  # ensures a new run every time you run this cell
        config={
            "proj_dim": proj_dim,
            "gru_hidden": hidden_size,
            "gru_layers": nlayer,
            "dropout_input": dropout_p,
            "batch_size": batchsize,
            "learning_rate": 0.001,
            "weight_decay": 0.01,
            "epochs": EPOCHS,
            "loss_at_last_timestep": True,
            "rnn_params": rnn_params,
            "classifier_params": classifier_params,
            "patience": patience,
            "time bins": "absolute"
        }
    )
    
    config = wandb.config
    
    # ------------------------------- setup model
    
    model = Model(
        rnn_params, 
        classifier_params, 
         proj_dim=proj_dim,
        dropout_p=dropout_p,
       loss_at_last_timestep=True, 
        device=DEVICE,
        patience=patience
    )
    
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), weight_decay=0.01, lr=0.001)
    
    dloader_train = DataLoader(  
        dset_train,
        batch_size=batchsize,
        shuffle=True,
    )
    
    dloader_test = DataLoader(  
        dset_test,
        batch_size=batchsize,
        shuffle=False,
    )

    # ------------------------------- train model
    
    model.train_model(dloader_train, dloader_test, EPOCHS, optimizer)
    wandb.finish()   
    
    acc_for_fold = model.best_val
    perf_per_split_abs_t.append(acc_for_fold)


In [ ]:
perf_per_split_abs_t = np.array(perf_per_split_abs_t)
print(perf_per_split_abs_t)

In [ ]:
save_as_pickle(path.join(savedir, 'perf_abs_t.pkl'), perf_per_split_abs_t)
save_as_pickle(path.join(savedir, 'config_abs_t.pkl'), config)


# Control: rerun with shuffled time during training

In [ ]:
decode_cfg['standardize_data'] = 1

time, X, y, groups, im_number, sess_idx, info, data_strs, h5_handle = load_data(
    data_cfg, root='..'
)
roi_str, arr_str, sess_str = data_strs
X, groups = process_data(X, sess_idx, im_number, groups, decode_cfg)

n_channels = X.shape[1]

tmask = np.isin(time, decoding_ts)
Xsel = X[..., tmask]
Xsel = np.moveaxis(Xsel, 1,2)
print(Xsel.shape)

In [ ]:
# compute splits and train
cv = crossvalidation.ImageStratifiedShuffleSplit(5, random_state=42)

perf_per_split_shuffle_t = []
for cv_split_idx, (train_idx, test_idx) in enumerate(cv.split(Xsel, y, groups=groups)):

    print("\n" * 10)
    print(f"--------------- STARTING SPLIT {cv_split_idx + 1}")

    X_train = Xsel[train_idx]  
    X_test = Xsel[test_idx]
    
    dset_train = TrialDataset(X_train, y[train_idx], shuffle_time=True)
    dset_test = TrialDataset(X_test, y[test_idx])

    # ---------------------------  Initialize a new WandB run
    
    wandb.init(
        project="time-series-classification",
        reinit=True,  # ensures a new run every time you run this cell
        config={
            "proj_dim": proj_dim,
            "gru_hidden": hidden_size,
            "gru_layers": nlayer,
            "dropout_input": dropout_p,
            "batch_size": batchsize,
            "learning_rate": 0.001,
            "weight_decay": 0.01,
            "epochs": EPOCHS,
            "loss_at_last_timestep": True,
            "rnn_params": rnn_params,
            "classifier_params": classifier_params,
            "patience": patience,
            "time bins": "shuffle"
        }
    )
    
    config = wandb.config
    
    # ------------------------------- setup model
    
    model = Model(
        rnn_params, 
        classifier_params, 
         proj_dim=proj_dim,
        dropout_p=dropout_p,
       loss_at_last_timestep=True, 
        device=DEVICE,
        patience=patience
    )
    
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), weight_decay=0.01, lr=0.001)
    
    dloader_train = DataLoader(  
        dset_train,
        batch_size=batchsize,
        shuffle=True,
    )
    
    dloader_test = DataLoader(  
        dset_test,
        batch_size=batchsize,
        shuffle=False,
    )

    # ------------------------------- train model
    
    model.train_model(dloader_train, dloader_test, EPOCHS, optimizer)
    wandb.finish()   
    
    acc_for_fold = model.best_val
    perf_per_split_shuffle_t.append(acc_for_fold)


In [ ]:
perf_per_split_shuffle_t = np.array(perf_per_split_shuffle_t)
print(perf_per_split_shuffle_t)

In [ ]:
save_as_pickle(path.join(savedir, 'perf_shuffle_t.pkl'), perf_per_split_shuffle_t)
save_as_pickle(path.join(savedir, 'config_shuffle_t.pkl'), config)


# Control: rerun with repeated feedforward avg. data

In [ ]:
decoding_ts_avg = np.arange(70, 171, 10)
tmask = np.isin(time, decoding_ts_avg)
print(tmask.sum())

Xsel = X[..., tmask]
Xsel = Xsel.mean(axis=-1, keepdims=True)
Xsel = np.repeat(Xsel, len(decoding_ts), axis=-1) # repeat to match time series for abs_t
Xsel = np.moveaxis(Xsel, 1,2)
print(Xsel.shape)

In [ ]:
# compute splits and train
cv = crossvalidation.ImageStratifiedShuffleSplit(5, random_state=42)

perf_per_split_ff_avg_t = []

for cv_split_idx, (train_idx, test_idx) in enumerate(cv.split(Xsel, y, groups=groups)):

    print("\n" * 10)
    print(f"--------------- STARTING SPLIT {cv_split_idx + 1}")

    X_train = Xsel[train_idx]  
    X_test = Xsel[test_idx]
    
    dset_train = TrialDataset(X_train, y[train_idx])
    dset_test = TrialDataset(X_test, y[test_idx])

    # ---------------------------  Initialize a new WandB run
    
    wandb.init(
        project="time-series-classification",
        reinit=True,  # ensures a new run every time you run this cell
        config={
            "proj_dim": proj_dim,
            "gru_hidden": hidden_size,
            "gru_layers": nlayer,
            "dropout_input": dropout_p,
            "batch_size": batchsize,
            "learning_rate": 0.001,
            "weight_decay": 0.01,
            "epochs": EPOCHS,
            "loss_at_last_timestep": True,
            "rnn_params": rnn_params,
            "classifier_params": classifier_params,
            "patience": patience,
            "time bins": "avg. ff"
        }
    )
    
    config = wandb.config
    
    # ------------------------------- setup model
    
    model = Model(
        rnn_params, 
        classifier_params, 
         proj_dim=proj_dim,
        dropout_p=dropout_p,
       loss_at_last_timestep=True, 
        device=DEVICE,
        patience=patience
    )
    
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), weight_decay=0.01, lr=0.001)
    
    dloader_train = DataLoader(  
        dset_train,
        batch_size=batchsize,
        shuffle=True,
    )
    
    dloader_test = DataLoader(  
        dset_test,
        batch_size=batchsize,
        shuffle=False,
    )

    # ------------------------------- train model
    
    model.train_model(dloader_train, dloader_test, EPOCHS, optimizer)
    wandb.finish()   
    
    acc_for_fold = model.best_val
    perf_per_split_ff_avg_t.append(acc_for_fold)

In [ ]:
perf_per_split_ff_avg_t = np.array(perf_per_split_ff_avg_t)
print(perf_per_split_ff_avg_t)

In [ ]:
save_as_pickle(path.join(savedir, 'perf_const_t.pkl'), perf_per_split_ff_avg_t)
save_as_pickle(path.join(savedir, 'config_const_t.pkl'), config)
